# Z4 115M — 10-minute Kaggle profile

This notebook runs the **115M** preset with the `kaggle10m` short-run profile:
same 115,238,237-parameter model geometry, 30-step warmup, no EMA, aux-head
compute skipped when its loss weight is zero, and a 600-second training cap.

The result is an early-training comparison, not a frontier claim.  See
`z3/KAGGLE_10M_ARCHITECTURE.md` for the profile and rationale.


In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator (T4)."
props = torch.cuda.get_device_properties(0)
print(props.name, round(props.total_memory / 1024**3, 1), "GiB")


In [ ]:
import os, subprocess
from pathlib import Path

REPO = Path("/kaggle/working/nanlm_experimental")
if not (REPO / "z3" / "train_v3.py").is_file():
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/shemshallah/nanlm_experimental.git",
                    str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
os.chdir(REPO)
print(REPO)


In [ ]:
# Kaggle usually has torch; add only the tokenizer/data packages.
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "tokenizers", "numpy", "pyarrow"], check=True)


In [ ]:
# Download the first 64 MiB of a public FineWeb-Edu shard.
raw = Path("fineweb_edu.txt")
if not raw.is_file() or raw.stat().st_size < 32 * 1024 * 1024:
    rc = subprocess.run([
        "curl", "-L", "--fail", "--retry", "3",
        "--range", "0-67108863",
        "-o", str(raw),
        "https://www.kaggle.com/api/v1/datasets/download/nameonlu/fineweb-edu/fineweb_edu_0000.txt",
    ]).returncode
    if rc != 0 or not raw.is_file() or raw.stat().st_size < 1024 * 1024:
        print("range download failed; falling back to the public HF parquet")
        subprocess.run([
            "curl", "-L", "--fail", "--retry", "3", "-o", "fineweb.parquet",
            "https://huggingface.co/datasets/HuggingFaceFW/fineweb-edu/resolve/main/sample/10BT/000_00000.parquet",
        ], check=True)
        subprocess.run([sys.executable, "z3/parquet_to_text.py",
                        "--input", "fineweb.parquet",
                        "--output", str(raw)], check=True)
print(raw, raw.stat().st_size)


In [ ]:
# Build document-framed uint16 bins and the manifest used by the 115M trainer.
subprocess.run([
    sys.executable, "z3/build_domain_bins.py",
    "--corpus", f"fineweb={raw}",
    "--out", "data/fineweb_bins",
    "--max-bytes-per-domain", "64000000",
    "--train-fraction", "0.98",
    "--valid-fraction", "0.01",
], check=True)


In [ ]:
# 10-minute training.  The preset caps wall time at 600 seconds.
OUT = Path("/kaggle/working/runs/z3_115m_kaggle10m")
cmd = [
    sys.executable, "z3/train_v3.py",
    "--preset", "kaggle10m",
    "--corpus", "data/fineweb_bins/train.bin",
    "--val-corpus", "data/fineweb_bins/valid.bin",
    "--data-manifest", "data/fineweb_bins/manifest.json",
    "--out", str(OUT),
]
print(" ".join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
# Held-out per-domain evaluation.  The builder writes data/fineweb_bins/test/.
subprocess.run([
    sys.executable, "z3/eval_domains.py",
    "--checkpoint", str(OUT / "best.pt"),
    "--test-dir", "data/fineweb_bins/test",
    "--data-manifest", "data/fineweb_bins/manifest.json",
    "--tokenizer", "layered",
    "--document-framing", "--sidecar", "--require-sidecar",
    "--ctx", "512", "--batch-size", "4", "--device", "cuda",
    "--out", str(OUT / "test_domains.json"),
], check=True)


In [ ]:
import json
result = json.loads((OUT / "result.json").read_text())
test = json.loads((OUT / "test_domains.json").read_text())
print(json.dumps({
    "params": result.get("params"),
    "steps": result.get("steps_completed"),
    "tokens_seen": result.get("tokens_seen"),
    "nonfinite_skips": result.get("nonfinite_skips"),
    "best_step": (result.get("best") or {}).get("step"),
    "best_bpb": ((result.get("best") or {}).get("metrics") or {}).get("bits_per_byte"),
    "final_bpb": (result.get("final") or {}).get("bits_per_byte"),
    "macro_test_bpb": test.get("macro_bits_per_byte"),
    "feedback": (result.get("final") or {}).get("feedback"),
}, indent=2, sort_keys=True))


In [ ]:
# Optional zero-shot WikiText-2 / WikiText-103.  Keep this out of the 10-minute
# training budget; run it only if the Kaggle session still has time.
# subprocess.run([
#     sys.executable, "z3/eval_v3.py",
#     "--checkpoint", str(OUT / "best.pt"),
#     "--corpus", "wt2=eval_data/wt2_test.txt",
#     "--corpus", "wt103=eval_data/wt103_test.txt",
#     "--tokenizer", "layered", "--ctx", "512", "--batches", "8",
#     "--batch-size", "4", "--device", "cuda",
#     "--out", str(OUT / "zero_shot.json"),
# ], check=True)
